# 10 · GSE232381 · bulk_RNA_seq · download

Chen YC et al. *Heliyon* 2024;10:e32303. Lupus nephritis, active vs inactive; NovaSeq 6000,
peripheral blood cells. GEO lists 25 samples.

The authors deposited an Excel file with, for each of the 25 samples, "Expected count", TPM and FPKM
values per gene. These notebooks use the authors' file.

| file | source |
|---|---|
| `GSE232381_Processed_data_files-Active_LN_vs_Inactive_LN.xlsx` | file the authors attached to the GEO record |
| `GSE232381_series_matrix.txt.gz` | GEO's file of sample information (the series matrix) |

## Download the authors' file and the series matrix

In [1]:
source("../src/paths.R")
options(timeout = 3600)
authors_file <- raw("GSE232381", "GSE232381_Processed_data_files-Active_LN_vs_Inactive_LN.xlsx")
if (!file.exists(authors_file))
  download.file("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE232nnn/GSE232381/suppl/GSE232381_Processed_data_files-Active_LN_vs_Inactive_LN.xlsx",
                authors_file, mode = "wb")
series <- raw("GSE232381", "GSE232381_series_matrix.txt.gz")
if (!file.exists(series))
  download.file("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE232nnn/GSE232381/matrix/GSE232381_series_matrix.txt.gz",
                series, mode = "wb")

**Explanation**
- `options(timeout = 3600)` and the download pattern as in notebook 00.
- The authors' file is attached to the GEO record; its full web address is given. `mode = "wb"` writes it byte for byte, which an Excel file needs.
- The series matrix holds the sample labels, as in notebook 00.

**Check.** The Excel file must start with the bytes 50 4b ("PK"), and the series matrix with 1f 8b (gzip),
not a web page saved in their place.

In [2]:
stopifnot(identical(readBin(authors_file, "raw", 2), as.raw(c(0x50, 0x4b))),
          identical(readBin(series, "raw", 2), as.raw(c(0x1f, 0x8b))))
data.frame(file = basename(c(authors_file, series)), KB = round(file.size(c(authors_file, series)) / 1e3))

file,KB
<chr>,<dbl>
GSE232381_Processed_data_files-Active_LN_vs_Inactive_LN.xlsx,10227
GSE232381_series_matrix.txt.gz,3


**Explanation**
- `readBin(file, "raw", 2)` reads the first two bytes of a file.
- An .xlsx file is a zip archive, and every zip archive starts with the bytes 50 4b, the letters "PK"
  (PKWARE. APPNOTE.TXT: .ZIP File Format Specification, local file header signature). Every gzip file starts with 1f 8b
  (Deutsch P. GZIP file format specification version 4.3. RFC 1952, 1996).
- `stopifnot(identical(...))` stops the notebook if a file does not start with its bytes, for example if a
  web page was saved instead.
- The table shows each file's size in kilobytes: `/ 1e3`.